In [0]:
 def processInvoiceAddendumDetails_LateCycleTruUp(df_CycleData_Consumer_CycleFlag,UpdatedBy="ADB_TruUp"):

    df_PatientClassification = (spark.read.table('Promotion.DIM_PatientClassification')
                                        .select('PatientClassificationUUID','PatientClassificationName','ListPrice','PromotionUUID',
                                        col('EffectiveDate').alias('PCL_EffectiveDate'),
                                        col('TerminationDate').alias('PCL_TerminationDate')        
                                                ))

    DF_FACT_InvoiceAddendumDetails = spark.read.table('Promotion.FACT_InvoiceAddendumDetails').filter('VersionId = 1')
    DF_FACT_ConsumerSubscription_UUID = spark.read.table('Promotion.FACT_ConsumerSubscription').filter('VersionId != 1')
    DF_FACT_ConsumerSubscription_NKEY = spark.read.table('Promotion.FACT_ConsumerSubscription').filter('VersionId = 1')
    
    df_CycleData_Consumer_CycleFlag_UniqueID = df_CycleData_Consumer_CycleFlag.select('PromotionUUID','CoolSculptingID','MaxInvoiceDate').drop_duplicates()
    

    df_Consumer_InvoiceDetails_TruUp = (DF_FACT_InvoiceAddendumDetails
                                        .join(df_CycleData_Consumer_CycleFlag_UniqueID,
                                                (DF_FACT_InvoiceAddendumDetails.CoolSculptingID == df_CycleData_Consumer_CycleFlag_UniqueID.CoolSculptingID)
                                                &
                                                (DF_FACT_InvoiceAddendumDetails.PromotionUUID == df_CycleData_Consumer_CycleFlag_UniqueID.PromotionUUID)
                                                ,'inner')
                                            .select(DF_FACT_InvoiceAddendumDetails.InvoiceAddendumDetailsUUID,
                                                    DF_FACT_InvoiceAddendumDetails.CycleDate,
                                                    DF_FACT_InvoiceAddendumDetails.CycleCount,
                                                    DF_FACT_InvoiceAddendumDetails.ShipToAccountUUID,
                                                    DF_FACT_InvoiceAddendumDetails.SoldToAccountID,
                                                    DF_FACT_InvoiceAddendumDetails.PromotionUUID,
                                                    DF_FACT_InvoiceAddendumDetails.CoolSculptingID,
                                                    DF_FACT_InvoiceAddendumDetails.IncrementalInvoiceCharge,
                                                    DF_FACT_InvoiceAddendumDetails.ConsumerSubscriptionUUID,
                                                    DF_FACT_InvoiceAddendumDetails.VersionID,
                                                    DF_FACT_InvoiceAddendumDetails.SmartCardSerialNumber,
                                                    df_CycleData_Consumer_CycleFlag_UniqueID.MaxInvoiceDate
                                                    ))

    df_Consumer_InvoiceDetails_TruUp = (df_Consumer_InvoiceDetails_TruUp
                                        .join(DF_FACT_ConsumerSubscription_UUID,(df_Consumer_InvoiceDetails_TruUp.ConsumerSubscriptionUUID == DF_FACT_ConsumerSubscription_UUID.ConsumerSubscriptionUUID),'inner')
                                        .join(DF_FACT_ConsumerSubscription_NKEY,    
                                                (df_Consumer_InvoiceDetails_TruUp.CoolSculptingID == DF_FACT_ConsumerSubscription_NKEY.CoolSculptingID) 
                                                &
                                            ((df_Consumer_InvoiceDetails_TruUp.ShipToAccountUUID == DF_FACT_ConsumerSubscription_NKEY.ShipToAccountUUID) 
                                                |
                                             ((df_Consumer_InvoiceDetails_TruUp.SoldToAccountID == DF_FACT_ConsumerSubscription_NKEY.SoldToAccountID) 
                                                &
                                              (df_Consumer_InvoiceDetails_TruUp.ShipToAccountUUID != DF_FACT_ConsumerSubscription_NKEY.ShipToAccountUUID)
                                             )
                                            )&
                                                (df_Consumer_InvoiceDetails_TruUp.PromotionUUID == DF_FACT_ConsumerSubscription_NKEY.PromotionUUID) 
                                                &
                                                (df_Consumer_InvoiceDetails_TruUp.CycleDate.between(DF_FACT_ConsumerSubscription_NKEY.SubscriptionStartDate,DF_FACT_ConsumerSubscription_NKEY.SubscriptionEndDate)),'left')
                                            .select(df_Consumer_InvoiceDetails_TruUp.InvoiceAddendumDetailsUUID,
                                                    df_Consumer_InvoiceDetails_TruUp.CycleDate,
                                                    df_Consumer_InvoiceDetails_TruUp.CycleCount,
                                                    df_Consumer_InvoiceDetails_TruUp.ShipToAccountUUID,
                                                    df_Consumer_InvoiceDetails_TruUp.SoldToAccountID,
                                                    df_Consumer_InvoiceDetails_TruUp.PromotionUUID,
                                                    df_Consumer_InvoiceDetails_TruUp.CoolSculptingID,
                                                    df_Consumer_InvoiceDetails_TruUp.IncrementalInvoiceCharge,
                                                    df_Consumer_InvoiceDetails_TruUp.VersionID,
                                                    df_Consumer_InvoiceDetails_TruUp.SmartCardSerialNumber,
                                                    df_Consumer_InvoiceDetails_TruUp.ApplicatorSerialNumber,
                                                    df_Consumer_InvoiceDetails_TruUp.CycleID,
                                                    DF_FACT_ConsumerSubscription_UUID.ConsumerSubscriptionUUID.alias('ConsumerSubscriptionUUID_1'),
                                                    DF_FACT_ConsumerSubscription_UUID.VersionID.alias('SubscriptionVersionID_1'),
                                                    DF_FACT_ConsumerSubscription_NKEY.ConsumerSubscriptionUUID.alias('ConsumerSubscriptionUUID_2'),
                                                    DF_FACT_ConsumerSubscription_NKEY.VersionID.alias('SubscriptionVersionID_2'),
                                                    DF_FACT_ConsumerSubscription_NKEY.SubscriptionStartDate.alias('SubscriptionStartDate_2'),
                                                    DF_FACT_ConsumerSubscription_NKEY.SubscriptionEndDate.alias('SubscriptionEndDate_2'),
                                                    DF_FACT_ConsumerSubscription_NKEY.Comments.alias('Comments_NKEY'),
                                                    df_Consumer_InvoiceDetails_TruUp.MaxInvoiceDate
                                                    )
                                            .filter('SubscriptionVersionID_1 != 1')
                                .withColumn('SubscriptionInvoiceAmt_1',sum('IncrementalInvoiceCharge').over(Window.partitionBy('ConsumerSubscriptionUUID_1')))
                                .withColumn('Prev_SubscriptionInvoiceAmt',sum('IncrementalInvoiceCharge').over(Window.partitionBy('ConsumerSubscriptionUUID_2'))))
    
    df_AccountFlags =  getAccountFlags(df_Consumer_InvoiceDetails_TruUp,v_flagType='All',v_flagSubType='PerSubscription')
    df_Consumer_InvoiceDetails_TruUp_Flag = df_Consumer_InvoiceDetails_TruUp.join(df_AccountFlags,['ShipToAccountUUID','PromotionUUID','CycleDate'],'left')

    df_Consumer_InvoiceDetails_TruUp_Classification = (df_Consumer_InvoiceDetails_TruUp_Flag
                            .withColumn('SubscriptionInvoiceAmt',when(col('Prev_SubscriptionInvoiceAmt').isNull(),lit(0))
                                                                .otherwise(col('Prev_SubscriptionInvoiceAmt')))
                            .withColumn('ROW_NUM',row_number().over(Window.partitionBy('ConsumerSubscriptionUUID_2','PromotionUUID')
                                                            .orderBy(asc('CycleDate'),asc('ShipToAccountUUID'))))
                            .withColumn('NewInvoiceAddendumDetailsUUID',expr('uuid()'))
                        )

    df_Consumer_InvoiceDetails_TruUp_Classification = applyRules(df_Consumer_InvoiceDetails_TruUp_Classification,'InvoiceAddendumDetails_TruUp')
    df_Consumer_InvoiceDetails_TruUp_Classification = df_Consumer_InvoiceDetails_TruUp_Classification.drop('SubscriptionInvoiceAmt','ROW_NUM','PerPatientFeeFlag','FollowUpVisitFlag','IncrementalInvoiceCharge')

    df_Consumer_InvoiceDetails_TruUp_Class = (df_Consumer_InvoiceDetails_TruUp_Classification
                                        .join(df_PatientClassification,['PatientClassificationName','PromotionUUID'],'inner')
                                        .filter("CycleDate between PCL_EffectiveDate AND PCL_TerminationDate")
                                        .drop('PCL_TerminationDate','PCL_EffectiveDate'))

    df_Consumer_InvoiceDetails_TruUp_Class_Insert = (df_Consumer_InvoiceDetails_TruUp_Class
                                                        .withColumn('InvoiceAddendumDetailsUUID',expr('uuid()'))
                                                        .withColumn('VersionID',lit(1))
                                                        .withColumn('CreatedBy',lit(UpdatedBy))
                                                        .withColumn('CreatedDate',lit(current_timestamp()))
                                                        .withColumn('UpdatedBy',lit(UpdatedBy))
                                                        .withColumn('UpdatedDate',lit(current_timestamp()))
                                                    )

    df_Consumer_InvoiceDetails_TruUp_Class_Update = (df_Consumer_InvoiceDetails_TruUp_Class
                                                        .withColumn('VersionID',col('VersionID')+1)
                                                        .withColumn('CreatedBy',lit(UpdatedBy))
                                                        .withColumn('CreatedDate',lit(current_timestamp()))
                                                        .withColumn('UpdatedBy',lit(UpdatedBy))
                                                        .withColumn('UpdatedDate',lit(current_timestamp()))
                                                    )

    df_Consumer_InvoiceDetails_TruUp_Final = df_Consumer_InvoiceDetails_TruUp_Class_Insert.unionByName(df_Consumer_InvoiceDetails_TruUp_Class_Update,allowMissingColumns=True)

    dl_FACTInvoiceAddendumDetails = DeltaTable.forName(spark, 'Promotion.FACT_InvoiceAddendumDetails')

    (dl_FACTInvoiceAddendumDetails.alias("tgt")
            .merge(df_Consumer_InvoiceDetails_TruUp_Final.alias("src"),
                ("src.InvoiceAddendumDetailsUUID = tgt.InvoiceAddendumDetailsUUID AND tgt.PromotionUUID = src.PromotionUUID"))
            .whenMatchedUpdate(set={
                    "tgt.VersionID": "src.VersionID",
                    "tgt.Comments": "src.Comments",
                    "tgt.UpdatedBy": "src.UpdatedBy",
                    "tgt.UpdatedDate": "src.UpdatedDate",
                    "tgt.SmartCardSerialNumber": "concat_ws(', ', array_distinct(split(concat_ws(', ', tgt.SmartCardSerialNumber, src.SmartCardSerialNumber), ', ')))"
            })
            .whenNotMatchedInsert(values =
                {
                    "tgt.InvoiceAddendumDetailsUUID": "src.NewInvoiceAddendumDetailsUUID",
                    "tgt.ConsumerSubscriptionUUID": "src.ConsumerSubscriptionUUID_2",
                    "tgt.CoolSculptingID": "src.CoolSculptingID",
                    "tgt.ShipToAccountUUID": "src.ShipToAccountUUID",
                    "tgt.SoldToAccountID": "src.SoldToAccountID",                    
                    "tgt.PromotionUUID": "src.PromotionUUID",
                    "tgt.CycleDate": "src.CycleDate",
                    "tgt.CycleCount": "src.CycleCount",
                    "tgt.IncrementalInvoiceCharge": "src.ListPrice",
                    "tgt.PatientClassificationUUID": "src.PatientClassificationUUID",
                    "tgt.VersionID": "src.VersionID",
                    "tgt.Comments": "src.Comments",
                    "tgt.CreatedBy": "src.CreatedBy",
                    "tgt.CreatedDate": "src.CreatedDate",
                    "tgt.UpdatedBy": "src.UpdatedBy",
                    "tgt.UpdatedDate": "src.UpdatedDate",
                    "tgt.SmartCardSerialNumber": "src.SmartCardSerialNumber",
                    "tgt.ApplicatorSerialNumber": "src.ApplicatorSerialNumber",
                    "tgt.CycleID": "src.CycleID"
                })
    .execute())

In [0]:
 def processInvoiceAddendumDetails_LateCycleTruUp_ConsumerBased(df_CycleData_Consumer_CycleFlag,UpdatedBy="ADB_TruUp"):


    df_PatientClassification = (spark.read.table('Promotion.DIM_PatientClassification')
                                        .select('PatientClassificationUUID','PatientClassificationName','ListPrice','PromotionUUID',
                                        col('EffectiveDate').alias('PCL_EffectiveDate'),
                                        col('TerminationDate').alias('PCL_TerminationDate')        
                                                ))

    DF_FACT_InvoiceAddendumDetails = spark.read.table('Promotion.FACT_InvoiceAddendumDetails').filter('VersionId = 1')
    DF_FACT_ConsumerSubscription_UUID = spark.read.table('Promotion.FACT_ConsumerSubscription').filter('VersionId != 1')
    DF_FACT_ConsumerSubscription_NKEY = spark.read.table('Promotion.FACT_ConsumerSubscription').filter('VersionId = 1')
    
    df_CycleData_Consumer_CycleFlag_UniqueID = df_CycleData_Consumer_CycleFlag.select('PromotionUUID','CoolSculptingID','MaxInvoiceDate').drop_duplicates()
    

    df_Consumer_InvoiceDetails_TruUp = (DF_FACT_InvoiceAddendumDetails
                                        .join(df_CycleData_Consumer_CycleFlag_UniqueID,
                                                (DF_FACT_InvoiceAddendumDetails.CoolSculptingID == df_CycleData_Consumer_CycleFlag_UniqueID.CoolSculptingID)
                                                &
                                                (DF_FACT_InvoiceAddendumDetails.PromotionUUID == df_CycleData_Consumer_CycleFlag_UniqueID.PromotionUUID)
                                                ,'inner')
                                            .select(DF_FACT_InvoiceAddendumDetails.InvoiceAddendumDetailsUUID,
                                                    DF_FACT_InvoiceAddendumDetails.CycleDate,
                                                    DF_FACT_InvoiceAddendumDetails.CycleCount,
                                                    DF_FACT_InvoiceAddendumDetails.ShipToAccountUUID,
                                                    DF_FACT_InvoiceAddendumDetails.SoldToAccountID,
                                                    DF_FACT_InvoiceAddendumDetails.PromotionUUID,
                                                    DF_FACT_InvoiceAddendumDetails.CoolSculptingID,
                                                    DF_FACT_InvoiceAddendumDetails.IncrementalInvoiceCharge,
                                                    DF_FACT_InvoiceAddendumDetails.ConsumerSubscriptionUUID,
                                                    DF_FACT_InvoiceAddendumDetails.VersionID,
                                                    DF_FACT_InvoiceAddendumDetails.SmartCardSerialNumber,
                                                    DF_FACT_InvoiceAddendumDetails.ApplicatorSerialNumber,
                                                    DF_FACT_InvoiceAddendumDetails.CycleID,
                                                    df_CycleData_Consumer_CycleFlag_UniqueID.MaxInvoiceDate
                                                    ))

    df_Consumer_InvoiceDetails_TruUp = (df_Consumer_InvoiceDetails_TruUp
                                        .join(DF_FACT_ConsumerSubscription_UUID,(df_Consumer_InvoiceDetails_TruUp.ConsumerSubscriptionUUID == DF_FACT_ConsumerSubscription_UUID.ConsumerSubscriptionUUID),'inner')
                                        .join(DF_FACT_ConsumerSubscription_NKEY,    
                                                (df_Consumer_InvoiceDetails_TruUp.CoolSculptingID == DF_FACT_ConsumerSubscription_NKEY.CoolSculptingID) 
                                                &
                                                (df_Consumer_InvoiceDetails_TruUp.PromotionUUID == DF_FACT_ConsumerSubscription_NKEY.PromotionUUID) 
                                                &
                                                (df_Consumer_InvoiceDetails_TruUp.CycleDate.between(DF_FACT_ConsumerSubscription_NKEY.SubscriptionStartDate,DF_FACT_ConsumerSubscription_NKEY.SubscriptionEndDate)),'left')
                                            .select(df_Consumer_InvoiceDetails_TruUp.InvoiceAddendumDetailsUUID,
                                                    df_Consumer_InvoiceDetails_TruUp.CycleDate,
                                                    df_Consumer_InvoiceDetails_TruUp.CycleCount,
                                                    df_Consumer_InvoiceDetails_TruUp.ShipToAccountUUID,
                                                    df_Consumer_InvoiceDetails_TruUp.SoldToAccountID,
                                                    df_Consumer_InvoiceDetails_TruUp.PromotionUUID,
                                                    df_Consumer_InvoiceDetails_TruUp.CoolSculptingID,
                                                    df_Consumer_InvoiceDetails_TruUp.IncrementalInvoiceCharge,
                                                    df_Consumer_InvoiceDetails_TruUp.VersionID,
                                                    df_Consumer_InvoiceDetails_TruUp.SmartCardSerialNumber,
                                                    df_Consumer_InvoiceDetails_TruUp.ApplicatorSerialNumber,
                                                    df_Consumer_InvoiceDetails_TruUp.CycleID,
                                                    DF_FACT_ConsumerSubscription_UUID.ConsumerSubscriptionUUID.alias('ConsumerSubscriptionUUID_1'),
                                                    DF_FACT_ConsumerSubscription_UUID.VersionID.alias('SubscriptionVersionID_1'),
                                                    DF_FACT_ConsumerSubscription_NKEY.ConsumerSubscriptionUUID.alias('ConsumerSubscriptionUUID_2'),
                                                    DF_FACT_ConsumerSubscription_NKEY.VersionID.alias('SubscriptionVersionID_2'),
                                                    DF_FACT_ConsumerSubscription_NKEY.SubscriptionStartDate.alias('SubscriptionStartDate_2'),
                                                    DF_FACT_ConsumerSubscription_NKEY.SubscriptionEndDate.alias('SubscriptionEndDate_2'),
                                                    DF_FACT_ConsumerSubscription_NKEY.Comments.alias('Comments_NKEY'),
                                                    df_Consumer_InvoiceDetails_TruUp.MaxInvoiceDate
                                                    )
                                            .filter('SubscriptionVersionID_1 != 1')
                                .withColumn('SubscriptionInvoiceAmt_1',sum('IncrementalInvoiceCharge').over(Window.partitionBy('ConsumerSubscriptionUUID_1')))
                                .withColumn('Prev_SubscriptionInvoiceAmt',sum('IncrementalInvoiceCharge').over(Window.partitionBy('ConsumerSubscriptionUUID_2'))))
    
    df_AccountFlags =  getAccountFlags(df_Consumer_InvoiceDetails_TruUp,v_flagType='All',v_flagSubType='PerSubscription')
    df_Consumer_InvoiceDetails_TruUp_Flag = df_Consumer_InvoiceDetails_TruUp.join(df_AccountFlags,['ShipToAccountUUID','PromotionUUID','CycleDate'],'left')

    df_Consumer_InvoiceDetails_TruUp_Classification = (df_Consumer_InvoiceDetails_TruUp_Flag
                            .withColumn('SubscriptionInvoiceAmt',when(col('Prev_SubscriptionInvoiceAmt').isNull(),lit(0))
                                                                .otherwise(col('Prev_SubscriptionInvoiceAmt')))
                            .withColumn('ROW_NUM',row_number().over(Window.partitionBy('ConsumerSubscriptionUUID_2','PromotionUUID')
                                                            .orderBy(asc('CycleDate'),asc('ShipToAccountUUID'))))
                            .withColumn('NewInvoiceAddendumDetailsUUID',expr('uuid()'))
                        )

    df_Consumer_InvoiceDetails_TruUp_Classification = applyRules(df_Consumer_InvoiceDetails_TruUp_Classification,'InvoiceAddendumDetails_TruUp')
    df_Consumer_InvoiceDetails_TruUp_Classification = df_Consumer_InvoiceDetails_TruUp_Classification.drop('SubscriptionInvoiceAmt','ROW_NUM','PerPatientFeeFlag','FollowUpVisitFlag','IncrementalInvoiceCharge')

    df_Consumer_InvoiceDetails_TruUp_Class = (df_Consumer_InvoiceDetails_TruUp_Classification
                                        .join(df_PatientClassification,['PatientClassificationName','PromotionUUID'],'inner')
                                        .filter("CycleDate between PCL_EffectiveDate AND PCL_TerminationDate")
                                        .drop('PCL_TerminationDate','PCL_EffectiveDate'))

    df_Consumer_InvoiceDetails_TruUp_Class_Insert = (df_Consumer_InvoiceDetails_TruUp_Class
                                                        .withColumn('InvoiceAddendumDetailsUUID',expr('uuid()'))
                                                        .withColumn('VersionID',lit(1))
                                                        .withColumn('CreatedBy',lit(UpdatedBy))
                                                        .withColumn('CreatedDate',lit(current_timestamp()))
                                                        .withColumn('UpdatedBy',lit(UpdatedBy))
                                                        .withColumn('UpdatedDate',lit(current_timestamp()))
                                                    )

    df_Consumer_InvoiceDetails_TruUp_Class_Update = (df_Consumer_InvoiceDetails_TruUp_Class
                                                        .withColumn('VersionID',col('VersionID')+1)
                                                        .withColumn('CreatedBy',lit(UpdatedBy))
                                                        .withColumn('CreatedDate',lit(current_timestamp()))
                                                        .withColumn('UpdatedBy',lit(UpdatedBy))
                                                        .withColumn('UpdatedDate',lit(current_timestamp()))
                                                    )

    df_Consumer_InvoiceDetails_TruUp_Final = df_Consumer_InvoiceDetails_TruUp_Class_Insert.unionByName(df_Consumer_InvoiceDetails_TruUp_Class_Update,allowMissingColumns=True)

    dl_FACTInvoiceAddendumDetails = DeltaTable.forName(spark, 'Promotion.FACT_InvoiceAddendumDetails')

    (dl_FACTInvoiceAddendumDetails.alias("tgt")
            .merge(df_Consumer_InvoiceDetails_TruUp_Final.alias("src"),
                ("src.InvoiceAddendumDetailsUUID = tgt.InvoiceAddendumDetailsUUID AND tgt.PromotionUUID = src.PromotionUUID"))
            .whenMatchedUpdate(set={
                    "tgt.VersionID": "src.VersionID",
                    "tgt.Comments": "src.Comments",
                    "tgt.UpdatedBy": "src.UpdatedBy",
                    "tgt.UpdatedDate": "src.UpdatedDate",
                    "tgt.SmartCardSerialNumber": "concat_ws(', ', array_distinct(split(concat_ws(', ', tgt.SmartCardSerialNumber, src.SmartCardSerialNumber), ', ')))"
            })
            .whenNotMatchedInsert(values =
                {
                    "tgt.InvoiceAddendumDetailsUUID": "src.NewInvoiceAddendumDetailsUUID",
                    "tgt.ConsumerSubscriptionUUID": "src.ConsumerSubscriptionUUID_2",
                    "tgt.CoolSculptingID": "src.CoolSculptingID",
                    "tgt.ShipToAccountUUID": "src.ShipToAccountUUID",
                    "tgt.SoldToAccountID": "src.SoldToAccountID",                    
                    "tgt.PromotionUUID": "src.PromotionUUID",
                    "tgt.CycleDate": "src.CycleDate",
                    "tgt.CycleCount": "src.CycleCount",
                    "tgt.IncrementalInvoiceCharge": "src.ListPrice",
                    "tgt.PatientClassificationUUID": "src.PatientClassificationUUID",
                    "tgt.VersionID": "src.VersionID",
                    "tgt.Comments": "src.Comments",
                    "tgt.CreatedBy": "src.CreatedBy",
                    "tgt.CreatedDate": "src.CreatedDate",
                    "tgt.UpdatedBy": "src.UpdatedBy",
                    "tgt.UpdatedDate": "src.UpdatedDate",
                    "tgt.SmartCardSerialNumber": "src.SmartCardSerialNumber",
                    "tgt.ApplicatorSerialNumber": "src.ApplicatorSerialNumber",
                    "tgt.CycleID": "src.CycleID"
                })
    .execute())